In [1]:
import pandas as pd
import numpy as np
import ast
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


In [2]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")


In [3]:
movies = movies.merge(credits, on="title")


In [4]:
movies = movies[["movie_id","title","genres","overview","keywords","cast","crew","popularity","budget","homepage","status","original_language","vote_average","release_date","runtime"]]
movies.dropna(inplace=True)


In [5]:
movies


,movie_id,title,genres,overview,keywords,cast,crew,popularity,budget,homepage,status,original_language,vote_average,release_date,runtime
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","In the 22nd century, a paraplegic Marine is di...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",150.437577,237000000,http://www.avatarmovie.com/,Released,en,7.2,2009-12-10,162.0
1,285,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","Captain Barbossa, long believed to be dead, ha...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",139.082615,300000000,http://disney.go.com/disneypictures/pirates/,Released,en,6.9,2007-05-19,169.0
2,206647,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",A cryptic message from Bond’s past sends him o...,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",107.376788,245000000,http://www.sonypictures.com/movies/spectre/,Released,en,6.3,2015-10-26,148.0
3,49026,The Dark Knight Rises,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",Following the death of District Attorney Harve...,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",112.312950,250000000,http://www.thedarkknightrises.com/,Released,en,7.6,2012-07-16,165.0
4,49529,John Carter,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","John Carter is a war-weary, former military ca...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",43.926995,260000000,http://movies.disney.com/john-carter,Released,en,6.1,2012-03-07,132.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4790,24055,The Puffy Chair,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 35, ""name...",Josh's life is pretty much in the toilet. He's...,"[{""id"": 171993, ""name"": ""mumblecore""}]","[{""cast_id"": 4, ""character"": ""Josh"", ""credit_i...","[{""credit_id"": ""52fe447fc3a368484e026881"", ""de...",1.243955,0,http://www.thepuffychairmovie.com,Released,en,6.2,2005-01-17,85.0
4797,157185,Tin Can Man,"[{""id"": 27, ""name"": ""Horror""}]",Recently dumped by his girlfirend for another ...,"[{""id"": 14903, ""name"": ""home invasion""}]","[{""cast_id"": 1, ""character"": ""Dave"", ""credit_i...","[{""credit_id"": ""54c7851b925141679100372a"", ""de...",0.332679,13,http://tincanmanthemovie.com/,Released,en,2.0,2007-01-01,84.0
4802,14337,Primer,"[{""id"": 878, ""name"": ""Science Fiction""}, {""id""...",Friends/fledgling entrepreneurs invent a devic...,"[{""id"": 1448, ""name"": ""distrust""}, {""id"": 2101...","[{""cast_id"": 1, ""character"": ""Aaron"", ""credit_...","[{""credit_id"": ""52fe45e79251416c75066791"", ""de...",23.307949,7000,http://www.primermovie.com,Released,en,6.9,2004-10-08,77.0
4806,231617,"Signed, Sealed, Delivered","[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...","""Signed, Sealed, Delivered"" introduces a dedic...","[{""id"": 248, ""name"": ""date""}, {""id"": 699, ""nam...","[{""cast_id"": 8, ""character"": ""Oliver O\u2019To...","[{""credit_id"": ""52fe4df3c3a36847f8275ecf"", ""de...",1.444476,0,http://www.hallmarkchannel.com/signedsealeddel...,Released,en,7.0,2013-10-13,120.0


In [6]:
# Convert JSON string into list of names
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i["name"])
    return L


In [7]:

# Keep top 3 cast members
def convertUpto3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter < 3:
            L.append(i["name"])
            counter += 1
        else:
            break
    return L


In [8]:

# Extract director
def fetch_director(obj):
    for i in ast.literal_eval(obj):
        if i['job'] == "Director":
            return [i['name']]
    return []


In [9]:
#cleaning data
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convertUpto3)
movies['crew'] = movies['crew'].apply(fetch_director)


In [10]:
# removeing spaces inside words for better matching
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

#splitting overview into list of words
movies['overview'] = movies['overview'].apply(lambda x:x.split())



In [11]:
# create a new column 'tags' which contains all the important information about a movie in a single column

movies['tags'] = movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew'] + movies['overview']

new_df = movies[['movie_id','title','tags']]
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


C:\Users\premk\AppData\Local\Temp\ipykernel_8792\1074930108.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


In [12]:
# tfidf vectorization 
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = tfidf.fit_transform(new_df['tags'])


In [13]:
#computing cosine similarity
similarity = linear_kernel(tfidf_matrix, tfidf_matrix)
similarity.shape


(1713, 1713)

In [14]:
def recommend(movie):
    # Normalize user input (ignore spaces + case)
    movie = movie.strip().lower()
    
    # Normalize titles in DataFrame
    titles_normalized = new_df['title'].str.strip().str.lower()
    
    if movie not in titles_normalized.values:
        print(f"Movie '{movie}' not found in database!")
        return []
    
    movie_index = titles_normalized[titles_normalized == movie].index[0]
    
    distances = similarity[movie_index]
    movie_list = sorted(
        list(enumerate(distances)), 
        reverse=True, key=lambda x: x[1]
    )[1:6]
    
    print(f"\n Top 5 recommendations for '{new_df.iloc[movie_index].title}':")
    recommendations = []
    for i in movie_list:
        rec_title = new_df.iloc[i[0]].title
        print(rec_title)
        recommendations.append(rec_title)
    
    return recommendations



In [15]:
recommend("Spider-Man 3")



 Top 5 recommendations for 'Spider-Man 3':
Spider-Man
Spider-Man 2
The Amazing Spider-Man 2
The Amazing Spider-Man
The Thing


['Spider-Man',
 'Spider-Man 2',
 'The Amazing Spider-Man 2',
 'The Amazing Spider-Man',
 'The Thing']

In [16]:
pickle.dump(movies, open('movies_data.pkl','wb'))
pickle.dump(similarity, open('similarity.pkl','wb'))
